In [ ]:
import os
import numpy as np


label_map = {
    "rest": 0,
    "motor": 1,
    "story_math": 2,
    "working_memory": 3
}

# Hyperparameters
window_size = 512
stride = 256
downsample_factor = 20
batch_size = 8


In [2]:
def extract_label_from_filename(filename):
    base = os.path.basename(filename).lower()
    if "rest" in base:
        return "rest"
    elif "motor" in base:
        return "motor"
    elif "story" in base or "math" in base:
        return "story_math"
    elif "working" in base or "memory" in base:
        return "working_memory"
    else:
        raise ValueError(f"Unknown label in {filename}")


In [3]:
import re

def infer_label_from_filename(filename, label_map):
    filename = filename.lower().replace('\\', '/')
    basename = os.path.basename(filename)
    
    for key in label_map:
        if key in basename:
            return label_map[key]
    
    raise ValueError(f"Could not infer label from filename: {filename}")

In [ ]:
import h5py
from sklearn.preprocessing import StandardScaler

def get_dataset_name(filepath):
    filename = os.path.basename(filepath)
    temp = filename.split(" ")[:-1]
    return " ".join(temp)

def load_and_preprocess(filepath, label_map, window_size=500, stride=500, downsample_factor=20):
    filename = filepath.lower()
    task_label = infer_label_from_filename(filepath, label_map)
    # Load data
    with h5py.File(filepath, 'r') as f:
        datasetname = list(f.keys())[0]
        data = f.get(datasetname)[()]  # Shape: (248, 35624)

    

    segments = []
    labels = []
    num_timepoints = data.shape[1]

    for start in range(0, num_timepoints - window_size + 1, stride):
        end = start + window_size
        window = data[:, start:end]

        mean = window.mean(axis=1, keepdims=True)
        std = window.std(axis=1, keepdims=True)
        window = (window - mean) / (std + 1e-8)

        segments.append(window[..., np.newaxis])  # Add channel dim for CNN
        labels.append(task_label)

    
    return segments, labels


In [5]:
import random

def data_generator(filepaths, batch_size, window_size, stride, downsample_factor, shuffle=True):
    while True:
        if shuffle:
            random.shuffle(filepaths)

        for filepath in filepaths:
            print(filepath)
            X, y = load_and_preprocess(filepath, label_map, window_size, stride, downsample_factor)
            indices = np.arange(len(X))
            if shuffle:
                np.random.shuffle(indices)
            for i in range(0, len(X), batch_size):
                batch_idx = indices[i:i+batch_size]
                X_np = np.array(X)
                y_np = np.array(y)
                yield X_np[batch_idx], y_np[batch_idx]


In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_cnn(input_shape=(248, 512, 1), num_classes=4):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    # model.add(Dropout(0.3))
    
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    # model.add(Dropout(0.3))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    # model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
import math

def train_model_in_chunks(model, train_files, batch_size, window_size, stride, downsample_factor,
                          chunk_size=8, epochs_per_chunk=1, total_outer_epochs=5):

    for outer_epoch in range(total_outer_epochs):
        print(f"\n{outer_epoch + 1}/{total_outer_epochs}")
        random.shuffle(train_files)

        for i in range(0, len(train_files), chunk_size):
            
            chunk_files = train_files[i:i + chunk_size]
            print(f" Training on chunk {i // chunk_size + 1}: {len(chunk_files)} files")
            total_segments = 0
            for f in chunk_files:
                with h5py.File(f, 'r') as h5f:
                    datasetname = list(h5f.keys())[0]
                    data = h5f[datasetname]
                    num_timepoints = data.shape[1]
                    n_segments = max(0, (num_timepoints - window_size) // stride + 1)
                    total_segments += n_segments
            steps_per_epoch = max(1, total_segments // batch_size)
            gen = data_generator(chunk_files, batch_size, window_size, stride, downsample_factor)

            model.fit(gen, steps_per_epoch=steps_per_epoch, epochs=epochs_per_chunk, verbose=1)


In [8]:
from glob import glob
# train_files = glob(r"./data/Cross/train/*.h5")
test_files = glob(r"./data/Cross/test1/*.h5")  # or test2, test3
data_dir='./data/Cross/train'
train_files = [
    os.path.normpath(os.path.join(data_dir, fname))
    for fname in os.listdir(data_dir)
    if fname.endswith('.h5')
]

model = build_cnn()
print(train_files)

train_model_in_chunks(model, train_files,
                      batch_size=batch_size,
                      window_size=window_size,
                      stride=stride,
                      downsample_factor=downsample_factor,
                      chunk_size=4,
                      epochs_per_chunk=1,
                      total_outer_epochs=5)




d:\projects\DL-Assignment-2\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


['data\\Cross\\train\\rest_113922_1.h5', 'data\\Cross\\train\\rest_113922_2.h5', 'data\\Cross\\train\\rest_113922_3.h5', 'data\\Cross\\train\\rest_113922_4.h5', 'data\\Cross\\train\\rest_113922_5.h5', 'data\\Cross\\train\\rest_113922_6.h5', 'data\\Cross\\train\\rest_113922_7.h5', 'data\\Cross\\train\\rest_113922_8.h5', 'data\\Cross\\train\\rest_164636_1.h5', 'data\\Cross\\train\\rest_164636_2.h5', 'data\\Cross\\train\\rest_164636_3.h5', 'data\\Cross\\train\\rest_164636_4.h5', 'data\\Cross\\train\\rest_164636_5.h5', 'data\\Cross\\train\\rest_164636_6.h5', 'data\\Cross\\train\\rest_164636_7.h5', 'data\\Cross\\train\\rest_164636_8.h5', 'data\\Cross\\train\\task_motor_113922_1.h5', 'data\\Cross\\train\\task_motor_113922_2.h5', 'data\\Cross\\train\\task_motor_113922_3.h5', 'data\\Cross\\train\\task_motor_113922_4.h5', 'data\\Cross\\train\\task_motor_113922_5.h5', 'data\\Cross\\train\\task_motor_113922_6.h5', 'data\\Cross\\train\\task_motor_113922_7.h5', 'data\\Cross\\train\\task_motor_11392

ResourceExhaustedError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
MemoryError: Unable to allocate 134. MiB for an array with shape (138, 248, 512, 1) and data type float64
Traceback (most recent call last):

  File "d:\projects\DL-Assignment-2\venv\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "d:\projects\DL-Assignment-2\venv\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "d:\projects\DL-Assignment-2\venv\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "d:\projects\DL-Assignment-2\venv\Lib\site-packages\keras\src\trainers\data_adapters\generator_data_

In [9]:
import h5py

# Count total number of test segments
def count_test_segments(test_files, window_size, stride, downsample_factor):
    total_segments = 0
    for f in test_files:
        with h5py.File(f, 'r') as h5f:
            datasetname = list(h5f.keys())[0]
            data = h5f[datasetname]
            num_timepoints = data.shape[1]
            n_segments = max(0, (num_timepoints - window_size) // stride + 1)
            total_segments += n_segments
    return total_segments

# Calculate total test segments and steps
total_test_segments = count_test_segments(test_files, window_size, stride, downsample_factor)
val_steps = total_test_segments // batch_size

# Create test generator (no shuffle!)
test_gen = data_generator(test_files, batch_size, window_size, stride, downsample_factor, shuffle=False)

# Evaluate
loss, acc = model.evaluate(test_gen, steps=val_steps, verbose=1)
print(f"Cross-subject test accuracy: {acc:.4f}")

./data/Cross/test1\rest_162935_1.h5
 16/276 ━━━━━━━━━━━━━━━━━━━━ 25s 98ms/step - accuracy: 0.0000e+00 - loss: 1.3776 ./data/Cross/test1\rest_162935_10.h5
 34/276 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.0000e+00 - loss: 1.3776./data/Cross/test1\rest_162935_3.h5
 52/276 ━━━━━━━━━━━━━━━━━━━━ 19s 87ms/step - accuracy: 0.0000e+00 - loss: 1.3776./data/Cross/test1\rest_162935_5.h5
 70/276 ━━━━━━━━━━━━━━━━━━━━ 17s 85ms/step - accuracy: 0.0000e+00 - loss: 1.3776./data/Cross/test1\task_motor_162935_1.h5
 89/276 ━━━━━━━━━━━━━━━━━━━━ 15s 83ms/step - accuracy: 0.0000e+00 - loss: 1.3785./data/Cross/test1\task_motor_162935_3.h5
105/276 ━━━━━━━━━━━━━━━━━━━━ 14s 83ms/step - accuracy: 0.0000e+00 - loss: 1.3799./data/Cross/test1\task_motor_162935_4.h5
125/276 ━━━━━━━━━━━━━━━━━━━━ 12s 82ms/step - accuracy: 0.0000e+00 - loss: 1.3819./data/Cross/test1\task_motor_162935_9.h5
142/276 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.0000e+00 - loss: 1.3835./data/Cross/test1\task_story_math_162935_2.h5